# Data Exploration — NVD + KEV + ATT&CK

This notebook provides exploratory analysis of the three primary data sources used by VulnInvestigator:

1. **NVD** (National Vulnerability Database) — CVSS-scored CVE records
2. **CISA KEV** (Known Exploited Vulnerabilities) — actively exploited CVEs
3. **MITRE ATT&CK** — adversary techniques and tactics

Run `scripts/run_pipeline.py` first to fetch and process the raw data.

In [ ]:
# Standard imports
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f'Project root: {PROJECT_ROOT}')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

In [ ]:
# Load processed data from Parquet (after pipeline run)
from src.pipeline.normaliser import DataNormaliser
from src.pipeline.database import Database
import yaml

with open(PROJECT_ROOT / 'config' / 'settings.yaml') as f:
    cfg = yaml.safe_load(f)

db_path = PROJECT_ROOT / cfg['data']['db_path']

if db_path.exists():
    with Database(str(db_path)) as db:
        rows = db.query('SELECT * FROM vulnerabilities')
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} rows from database')
    print(df.dtypes)
else:
    print(f'Database not found at {db_path}')
    print('Run: python scripts/run_pipeline.py')
    # Create a sample DataFrame for demonstration
    df = pd.DataFrame({
        'cve_id': [f'CVE-2024-{1000+i}' for i in range(20)],
        'cvss_score': np.random.uniform(1, 10, 20).round(1),
        'cvss_severity': np.random.choice(['CRITICAL','HIGH','MEDIUM','LOW'], 20),
        'in_kev': np.random.choice([0, 1], 20, p=[0.8, 0.2]),
        'risk_score': np.random.uniform(0, 1, 20).round(3),
        'risk_band': np.random.choice(['Critical','High','Medium','Low'], 20),
    })
    print('Using sample demonstration data.')

In [ ]:
# Basic statistics
print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print()
print('=== Column Types ===')
print(df.dtypes)
print()
print('=== Null Counts ===')
print(df.isnull().sum())
print()
print('=== Numeric Summary ===')
numeric_col = 'cvss_score' if 'cvss_score' in df.columns else 'risk_score'
print(df[[numeric_col]].describe())

In [ ]:
# CVSS score distribution
score_col = 'cvss_score' if 'cvss_score' in df.columns else 'cvss_v3_score'

fig = px.histogram(
    df,
    x=score_col,
    nbins=20,
    color='cvss_severity' if 'cvss_severity' in df.columns else None,
    color_discrete_map={
        'CRITICAL': '#d62728',
        'HIGH': '#ff7f0e',
        'MEDIUM': '#ffdd57',
        'LOW': '#2ca02c',
    },
    title='CVSS v3 Score Distribution',
    labels={score_col: 'CVSS v3 Base Score', 'count': 'Number of CVEs'},
    barmode='overlay',
    opacity=0.8,
)
fig.add_vline(x=9.0, line_dash='dash', line_color='red', annotation_text='CRITICAL (9.0)')
fig.add_vline(x=7.0, line_dash='dash', line_color='orange', annotation_text='HIGH (7.0)')
fig.update_layout(height=450)
fig.show()

In [ ]:
# KEV overlap analysis
if 'in_kev' in df.columns:
    kev_counts = df['in_kev'].value_counts().reset_index()
    kev_counts.columns = ['In KEV', 'Count']
    kev_counts['Label'] = kev_counts['In KEV'].map({1: 'In KEV (Exploited)', 0: 'Not in KEV'})

    fig_kev = px.pie(
        kev_counts,
        values='Count',
        names='Label',
        title='CVE KEV Status — Proportion Actively Exploited',
        color='Label',
        color_discrete_map={'In KEV (Exploited)': '#d62728', 'Not in KEV': '#aec7e8'},
    )
    fig_kev.show()

    # CVSS distribution within KEV vs non-KEV
    if score_col in df.columns:
        fig_box = px.box(
            df,
            x='in_kev',
            y=score_col,
            title='CVSS Score Distribution: KEV vs Non-KEV',
            labels={'in_kev': 'In KEV (1=Yes)', score_col: 'CVSS Score'},
        )
        fig_box.show()

        kev_mean = df[df['in_kev']==1][score_col].mean()
        non_kev_mean = df[df['in_kev']==0][score_col].mean()
        print(f'Mean CVSS (KEV):     {kev_mean:.2f}')
        print(f'Mean CVSS (Non-KEV): {non_kev_mean:.2f}')
        print(f'Difference:          {kev_mean - non_kev_mean:.2f}')
else:
    print('in_kev column not found — run the pipeline first.')

## Notes

### Key Observations
- The CVSS score distribution typically shows a bimodal pattern — many CVEs cluster at high severity (8–10) or low severity (1–4).
- KEV entries tend to have higher mean CVSS scores, confirming that CVSS is a useful but imperfect proxy for exploitation likelihood.
- The small proportion of CVEs in the KEV catalogue (~2–5%) highlights the importance of prioritising beyond raw CVSS score.

### Research Questions
1. **RQ1**: How much does contextual risk scoring change the ranking vs CVSS-only? (Measured by Kendall's τ)
2. **RQ2**: Can GenAI produce evidence-grounded investigation reports with low hallucination rates?
3. **RQ3**: What failure modes does the pipeline exhibit under adversarial or incomplete inputs?

### Next Steps
- Run `notebooks/02_scoring_comparison.ipynb` to compare CVSS vs contextual rankings.
- Run `scripts/run_evaluation.py` to generate quantitative evaluation metrics.